# Train the sentiment student models on a GPU

1. **Runtime → Change runtime type → T4 GPU → Save.**
2. Run the cells below in order (Shift+Enter). Cell 2 asks you to pick `sentiment_colab.zip`.
3. Cells 4 and 5 train two model sizes (12-layer: more accurate; 6-layer: 2× faster to serve).
4. The last cell downloads `student_v2.zip` (~90 MB, both models). Send that file back.

Expected time on a T4: about 30–40 minutes in total. Keep the browser tab open while it runs.

In [ ]:
# 1. Check that a GPU is attached (should list a Tesla T4)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Upload sentiment_colab.zip and unpack it
from google.colab import files
uploaded = files.upload()
!rm -rf /content/work && mkdir -p /content/work && unzip -q sentiment_colab.zip -d /content/work
!ls /content/work

In [ ]:
# 3. Install the two packages Colab doesn't have (torch and transformers are preinstalled)
!pip install -q onnx onnxruntime

In [ ]:
# Train the 12-layer multilingual MiniLM (128 tokens) on Codex labels (real posts) + TweetEval (human labels)
%cd /content/work
!SENTIMENT_DATA_ROOT=/content/work python -u scripts/train_student.py \n    --layers 12 --max-len 128 --epochs 3 --batch 32 --lr 5e-5 \n    --llm-labels data/silver/llm_codex.jsonl --silver-mode none \n    --out-dir /content/work/student12 2>&1 | grep --line-buffered -v -i warn

In [ ]:
# Train the 6-layer multilingual MiniLM (128 tokens) on Codex labels (real posts) + TweetEval (human labels)
%cd /content/work
!SENTIMENT_DATA_ROOT=/content/work python -u scripts/train_student.py \n    --layers 6 --max-len 128 --epochs 3 --batch 32 --lr 5e-5 \n    --llm-labels data/silver/llm_codex.jsonl --silver-mode none \n    --out-dir /content/work/student6 2>&1 | grep --line-buffered -v -i warn

In [ ]:
# Package both trained models and download them
!cd /content/work && zip -q -r /content/student_v2.zip student12/config.json student12/model.int8.onnx student12/tokenizer.json student6/config.json student6/model.int8.onnx student6/tokenizer.json
!ls -la /content/student_v2.zip
files.download('/content/student_v2.zip')